# Model training Iteration 1
##### In the Keras TextVectorization class 'output_mode' is set at 'tf-idf'

- Model 1: Model architecture based on example in https://www.geeksforgeeks.org/nlp/rnn-for-text-classifications-in-nlp/
- Model 2: The same architecture as Model 1 with an additional bidirectional layer of 128 units added.

In [1]:
# Read necessary modules
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import os
import tensorflow as tf
import re
import nltk
from nltk.corpus import stopwords
from collections import Counter

2025-11-20 13:48:32.567903: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763646513.031427      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763646513.159011      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Read dataset
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
# Function to write model training results to an external location 
def write_away(name, model_results, path):

# Writing away the results 
    name = pd.DataFrame.from_dict(model_results)
    name.to_csv(path, index=False)
    return name

In [4]:
# Function to remove integers from selected columns
def remove_numbers(x):
    return re.sub(r'\d+', '', x)

In [5]:
# Remove integers from column text
train_df['text'] = train_df['text'].apply(remove_numbers)
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,", people receive #wildfires evacuation orders ...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M. [: UTC]?km S of Volcano Hawaii. http://t.co...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [6]:
# function to remove stopwords from nltk corpus from strings
def basic_clean(x):
  """
  A simple function to clean up the data. All the words that
  are not designated as a stop word is then lemmatized after
  encoding and basic regex parsing are performed.
  """
  stops = set(stopwords.words('english'))
  words = re.sub(r'[^\w\s]', '', x).lower().split()
  return [word for word in words if word not in stops]

In [7]:
# apply function basic_clean to column text
train_df['text'] = train_df['text'].apply(basic_clean)

# apply lambda function to put column text in right format
train_df['text'] = train_df['text'].apply(lambda x: ', '.join(x))
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,"deeds, reason, earthquake, may, allah, forgive...",1
1,4,NaN,NaN,"forest, fire, near, la, ronge, sask, canada",1
2,5,NaN,NaN,"residents, asked, shelter, place, notified, of...",1
3,6,NaN,NaN,"people, receive, wildfires, evacuation, orders...",1
4,7,NaN,NaN,"got, sent, photo, ruby, alaska, smoke, wildfir...",1
...,...,...,...,...,...
7608,10869,NaN,NaN,"two, giant, cranes, holding, bridge, collapse,...",1
7609,10870,NaN,NaN,"aria_ahrary, thetawniest, control, wild, fires...",1
7610,10871,NaN,NaN,"utckm, volcano, hawaii, httptcozdtoydebj",1
7611,10872,NaN,NaN,"police, investigating, ebike, collided, car, l...",1


In [8]:
# Put text and target variable in an array
x_train = np.array(train_df.text)
y_train = np.array(train_df.target)

In [9]:
# Tokenize the text 
encoder_tf = tf.keras.layers.TextVectorization(
    max_tokens=2673,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=1,
    output_mode='tf-idf',
    output_sequence_length=None,
    pad_to_max_tokens=True,
    vocabulary=None,
    idf_weights= None,
    sparse=False,
    ragged=False,
    encoding='utf-8',
    name=None
)
encoder_tf

I0000 00:00:1763646604.123012      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1763646604.123689      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


<TextVectorization name=text_vectorization, built=False>

In [10]:
# Put train_df.text in tensorflow format
text_dataset = tf.data.Dataset.from_tensor_slices(train_df.text)
# Use encoder_tf to create the vocabulary 
encoder_tf.adapt(text_dataset)

### Model 1

In [14]:
# Define architecture Model 1 
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(1,), dtype = tf.string),
    encoder_tf,
    tf.keras.layers.Embedding(input_dim = len(encoder_tf.get_vocabulary()), output_dim = 64, mask_zero=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 2673)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 2673, 64)       │       171,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 2673, 128)      │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,561 (1.08 MB)

 Trainable params: 282,561 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# training Model 1
epochs = 20
batch_size = 150
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy", "auc"])

history = model.fit(x_train, y_train, batch_size= batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20


I0000 00:00:1763646762.919580     112 cuda_dnn.cc:529] Loaded cuDNN version 90300


46/46 ━━━━━━━━━━━━━━━━━━━━ 28s 414ms/step - accuracy: 0.5725 - auc: 0.5000 - loss: 6.8907 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 2/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 405ms/step - accuracy: 0.5708 - auc: 0.5000 - loss: 6.9184 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 3/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 412ms/step - accuracy: 0.5725 - auc: 0.5000 - loss: 6.8906 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 4/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 413ms/step - accuracy: 0.5743 - auc: 0.5000 - loss: 6.8607 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 5/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 407ms/step - accuracy: 0.5688 - auc: 0.5000 - loss: 6.9498 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 6/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 406ms/step - accuracy: 0.5809 - auc: 0.5000 - loss: 6.7545 - val_accuracy: 0.5341 - val_auc: 0.5000 - val_loss: 7.5091
Epoch 7/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 406

In [ ]:
# Model 1: Writing away the results 
write_away('BL_RNN', history.history, 'BL_RNN.csv')

### Model 2

In [16]:
# Define architecture Model 2 
model_II = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(1,), dtype = tf.string),
    encoder_tf,
    tf.keras.layers.Embedding(input_dim = len(encoder_tf.get_vocabulary()), output_dim = 64, mask_zero=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences = True)), 
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True)),  
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),  
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_II.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 2673)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 2673, 64)       │       171,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 2673, 256)      │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 2673, 128)      │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 578,497 (2.21 MB)

 Trainable params: 578,497 (2.21 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# training Model 2
epochs = 20
batch_size = 150
model_II.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy", "auc"])

history_II = model_II.fit(x_train, y_train, batch_size= batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 911ms/step - accuracy: 0.5417 - auc: 0.4943 - loss: 0.8502 - val_accuracy: 0.5341 - val_auc: 0.5397 - val_loss: 0.7048
Epoch 2/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 877ms/step - accuracy: 0.5719 - auc: 0.4908 - loss: 0.6868 - val_accuracy: 0.5341 - val_auc: 0.5597 - val_loss: 0.6970
Epoch 3/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 879ms/step - accuracy: 0.5718 - auc: 0.5002 - loss: 0.6835 - val_accuracy: 0.5341 - val_auc: 0.5608 - val_loss: 0.7020
Epoch 4/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 881ms/step - accuracy: 0.5689 - auc: 0.4982 - loss: 0.6868 - val_accuracy: 0.5341 - val_auc: 0.5967 - val_loss: 0.6910
Epoch 5/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 877ms/step - accuracy: 0.5715 - auc: 0.5100 - loss: 0.6831 - val_accuracy: 0.5341 - val_auc: 0.6002 - val_loss: 0.6891
Epoch 6/20
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 878ms/step - accuracy: 0.5639 - auc: 0.5207 - loss: 0.6850 - val_accuracy: 0.5341 - val_auc: 0.6000 - val_loss: 0.6944
Epoch 7/20
46/46 ━━━━━━━━━━━━━━━━━

In [ ]:
# Model 2: Writing away the results 
write_away('BL_RNN_II', history_II.history, 'BL_RNN_II.csv')